# Second data collection round

Extends the pilot to 15 apps (12 platforms + 3 D2C brands) to grow the labelled set and add more product-focused reviews. Excludes reviews already present in the pilot set.

## Setup

In [ ]:
!pip install google-play-scraper -q

## Load prior pilot IDs
Used to filter out duplicates from this round.

In [ ]:
from google.colab import files
import pandas as pd
uploaded = files.upload()
pilot = pd.read_csv("pilot_reviews.csv")
PILOT_IDS = set(pilot["review_id"])
print("Pilot reviews to skip:", len(PILOT_IDS))

## App list

In [ ]:
from google_play_scraper import app, reviews, search, Sort
import time, re

# name: (Play Store ID or None to search by name, group)
APPS = {
    # shopping and quick-commerce platforms
    "Meesho":            ("com.meesho.supply", "platform"),
    "Nykaa":             ("com.fsn.nykaa", "platform"),
    "Blinkit":           ("com.grofers.customerapp", "platform"),
    "Zepto":             ("com.zeptoconsumerapp", "platform"),
    "Flipkart":          ("com.flipkart.android", "platform"),
    "Amazon India":      ("in.amazon.mShop.android.shopping", "platform"),
    "Myntra":            ("com.myntra.android", "platform"),
    "AJIO":              ("com.ril.ajio", "platform"),
    "BigBasket":         ("com.bigbasket.mobileapp", "platform"),
    "Swiggy":            ("in.swiggy.android", "platform"),
    "Snapdeal":          ("com.snapdeal.main", "platform"),
    "Shopsy":            ("com.flipkart.shopsy", "platform"),
    # D2C brands (product-focused reviews)
    "Mamaearth":         ("com.bemamaearth.app", "d2c"),
    "Bewakoof":          ("com.bewakoof.bewakoof", "d2c"),
    "Lenskart":          ("com.lenskart.app", "d2c"),
    "Purplle":           ("com.manash.purplle", "d2c"),
}

RESOLVED = {}
for name, (app_id, group) in APPS.items():
    try:
        if app_id is None:
            hits = search(name, lang="en", country="in", n_hits=1)
            app_id = hits[0]["appId"] if hits else None
        info = app(app_id, lang="en", country="in")
        RESOLVED[name] = (app_id, group)
        print(f"OK    {name:18s} {app_id:40s} title: {info.get('title')} | reviews: {info.get('reviews')}")
    except Exception as e:
        print(f"FAIL  {name:18s} {app_id} ({e.__class__.__name__}) -> skipped")
print(f"\n{len(RESOLVED)} apps ready")

## Collect reviews
Per app: 5 star ratings x 2 language settings x 2 sort orders.

In [ ]:
PER_STAR = 150
SORTS = {"newest": Sort.NEWEST, "relevant": Sort.MOST_RELEVANT}

rows = []
for name, (app_id, group) in RESOLVED.items():
    got = 0
    for lang in ["en", "hi"]:
        for sort_name, sort in SORTS.items():
            for star in [1, 2, 3, 4, 5]:
                try:
                    result, _ = reviews(app_id, lang=lang, country="in", sort=sort,
                                        count=PER_STAR, filter_score_with=star)
                except Exception as e:
                    continue
                for r in result:
                    rows.append({"app": name, "group": group, "review_id": r["reviewId"],
                                 "text": r["content"], "stars": r["score"], "date": r["at"],
                                 "lang_setting": lang, "sort": sort_name})
                got += len(result)
                time.sleep(0.5)
    print(f"{name:18s} {got} raw reviews")
raw = pd.DataFrame(rows)
print("\nTotal raw:", len(raw))

## Cleaning and Hinglish detection

In [ ]:
import re
# Common Hindi words written in English letters, including the spelling variants we saw in the pilot data.
HINDI_WORDS = set("""
hai hain h nahi nahin nhi nai bahut bhut bhot bohot bohat bht accha acha achha achchha achcha acchi achhi achi
bekar bekaar bakwas bakwaas ghatiya faltu kya kyu kyun kyon kab kaise kaisa kaisi kuch koi sab sabse
mujhe muje mera meri mere hamara hamare hum ham aap apna apni aur bhi toh tha thi raha rahi rahe
gaya gayi gaye gye hua hui hue hota hoti hote diya diye liya liye kar karo karna karte karta karti kiya
wala wali wale lekin magar abhi jaldi der saman samaan kapda kapde bilkul ekdam yaar bhai sahi galat
paisa paise rupaye wapas vapas mila mile milta milti milte chahiye chaiye isme isse iska iski uska uski
unka yeh ye woh vo wo jo jab tak se ki ka ke ko pe par mein me mai main hu hun ho gya gyi kr krna krte
""".split())
AMBIGUOUS = {"h", "me", "par", "se", "ho", "hu", "ye", "vo", "wo", "jo", "ki", "ka", "ke", "ko", "pe", "mai", "main", "tak"}

def classify(text):
    text = str(text)
    if re.search(r"[\u0900-\u097F]", text):
        return "devanagari"
    words = re.findall(r"[a-z]+", text.lower())
    strong = sum(w in HINDI_WORDS and w not in AMBIGUOUS for w in words)
    weak = sum(w in AMBIGUOUS for w in words)
    score = strong + 0.5 * weak
    if score >= 2:
        return "hinglish"
    if score >= 1:
        return "maybe"
    return "english"


df = raw.drop_duplicates("review_id")
df = df[~df.review_id.isin(PILOT_IDS)]
df = df[df.text.notna()]
df = df.drop_duplicates("text")
df["n_words"] = df.text.str.split().str.len()
df = df[df.n_words.between(4, 150)].copy()
df["type"] = df.text.apply(classify)
print("Clean new reviews:", len(df))
print(df.type.value_counts())

## Summary statistics

In [ ]:
cm = df[df.type.isin(["hinglish", "maybe"])]
print("=== Code-mixed (hinglish + maybe) per app ===")
t = df.groupby(["group", "app"]).agg(clean=("text", "size"),
      code_mixed=("type", lambda s: s.isin(["hinglish", "maybe"]).sum()))
t["share"] = (t.code_mixed / t.clean).round(2)
print(t.sort_values("code_mixed", ascending=False).to_string())
print("\n=== Code-mixed per star ===")
print(cm.stars.value_counts().sort_index().to_string())
print("\n=== Code-mixed per group ===")
print(cm.group.value_counts().to_string())
print("\nTotal code-mixed candidates:", len(cm))

## Export

In [ ]:
df.to_csv("new_reviews_all.csv", index=False)
cm.to_csv("new_hinglish_candidates.csv", index=False)
files.download("new_hinglish_candidates.csv")
files.download("new_reviews_all.csv")